<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_dual_compact_sts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT dual-source — compact shake+TinyStories

Slim training notebook for a **compact** dual model trained on shake + TinyStories at char level. Same winning recipe as the wiki runs — `alt_mixed` batch mode + Uniform alpha + two AdamW optimizers, pass-level alternation — but on the new `shakespeare_tinystories_char` dataset and with a smaller model (n_layer=4, n_head=4, n_embd=256, ~6M params total) tuned to TinyStories being simpler than wiki.

**Just the training**, no analysis prelude. Hit Run All.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet zstandard tiktoken

In [ ]:
import os
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT

In [ ]:
!python data/shakespeare_tinystories_char/prepare.py

## Run directory (tagged `-compact-sts` so this run is distinct from the wiki runs)

In [ ]:
RUN_ID = None   # set explicitly to resume, e.g. "20260519-XXXXXX-compact-sts"

In [ ]:
import time, os, glob
DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if 'RUN_ID' not in dir() or RUN_ID is None:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-compact-sts'
elif not RUN_ID.endswith('-compact-sts'):
    RUN_ID = RUN_ID + '-compact-sts'
OUT_DIR_DUAL = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR_DUAL, exist_ok=True)
print('run dir:', OUT_DIR_DUAL)
snaps = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, '*.pt.zst')))
print(f'  {len(snaps)} existing snapshots' + (' (will resume)' if snaps else ' (fresh run)'))

## Train

In [ ]:
!python train_dual.py config/train_shakespeare_wiki_dual_alt_mixed_compact_sts.py \
    --out_dir=$OUT_DIR_DUAL \
    --batch_mode=alt_mixed \
    --first_pass_corpus=shake \
    --mix_distribution=uniform